# 05 - Error Analysis & Blocking Improvement

**Tujuan:** menganalisis kenapa REVIEW selalu 0, mengidentifikasi masalah candidate set yang terlalu homogen, lalu memperbaiki dengan melebarkan blocking rule (satu komponen saja).

**Data masukan:**
- `customers_standarized.csv` (50.000 rows)
- `splink_predictions.csv` dari Nb04
- `splink_entities.csv` dari Nb04
- Reference positive labels (same `customer_id`, 1.867 pairs)

**Pipeline Nb05:**
1. Setup & load data
2. Rekap baseline & error matrix
3. Root cause analysis: gap weight
4. Problem identification: REVIEW=0 dan single negative candidate
5. Improvement attempt: lebarkan blocking (1 komponen)
6. Rebuild model + predict
7. Decision & evaluasi
8. Comparasi baseline vs improved
9. Entity mapping
10. Visualisasi
11. Summary & rekomendasi Nb06

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import duckdb
import splink.comparison_library as cl
import splink.blocking_rule_library as br
from splink import DuckDBAPI, Linker, SettingsCreator

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW_PATH     = r"C:\Users\User\Downloads\Fix\data\raw\customers_standarized.csv"
PRED_BASE    = r"C:\Users\User\Downloads\Fix\data\raw\splink_predictions.csv"
PRED_RELAXED = r"C:\Users\User\Downloads\Fix\data\raw\splink_predictions_relaxed.csv"
ENTITY_RELAX = r"C:\Users\User\Downloads\Fix\data\raw\splink_entities_relaxed.csv"
print("Setup OK")

Setup OK


## 2. Load Data & Reference Labels

In [2]:
df = pd.read_csv(RAW_PATH, dtype=str)
df = df.reset_index().rename(columns={"index": "unique_id"})

def norm(s):
    return s.astype(str).str.lower().str.strip().str.replace(r"\s+", " ", regex=True)

df["city_std"] = norm(df["city"])
df["device_id_std"] = df["device_id(s)"].astype(str)
print(f"Loaded: {len(df):,} rows x {df.shape[1]} cols")

Loaded: 50,000 rows x 26 cols


In [3]:
cid_groups = {}
for idx in range(len(df)):
    cid = str(df.iloc[idx]["customer_id"])
    cid_groups.setdefault(cid, []).append(idx)

ref_pos = set()
for idxs in cid_groups.values():
    if len(idxs) < 2:
        continue
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            ref_pos.add((int(idxs[i]), int(idxs[j])))

print(f"Reference positive pairs: {len(ref_pos):,}")

Reference positive pairs: 1,867


## 3. Rekap Baseline (Nb04)

Fakta dari Nb04:
- Candidate set: 1.868 pasangan (hampir semua duplikat asli)
- 1.867 same-customer (positive), 1 negative
- Precision 1.0 / Recall 1.0 — tapi ini mengukur separasi sempurna, bukan kualitas dunia riil
- REVIEW band: 0 pasangan

In [4]:
pred_base = pd.read_csv(PRED_BASE)
pred_base["unique_id_l"] = pred_base["unique_id_l"].astype(int)
pred_base["unique_id_r"] = pred_base["unique_id_r"].astype(int)

cand_strict = set(zip(pred_base["unique_id_l"], pred_base["unique_id_r"]))
coverage_strict = len(ref_pos & cand_strict)

print(f"Baseline candidates   : {len(cand_strict):,}")
print(f"Positives covered     : {coverage_strict:,} / {len(ref_pos):,} = {coverage_strict/len(ref_pos)*100:.2f}%")
print(f"Negative candidates   : {len(cand_strict - ref_pos):,}")
print()
print("Decision distribution:")
print(pred_base["decision"].value_counts().to_string())

Baseline candidates   : 1,868
Positives covered     : 1,867 / 1,867 = 100.00%
Negative candidates   : 1

Decision distribution:
decision
MATCH        1867
NON-MATCH       1


## 4. Root Cause: Gap Weight

Distribusi `match_weight` antara positive dan negative menentukan apakah REVIEW band terisi.

**Gap yang dibutuhkan untuk REVIEW:**
- Band REVIEW = `match_weight` dalam [-2, +2] (probability 0.20-0.85)
- Panjang band = 4 poin weight

**Fakta dari baseline:**
- Positive min weight: +31.49
- Negative max weight: -170.32
- Gap: 201.81 poin (50x lebih lebar dari band REVIEW)

**Kesimpulan:** REVIEW=0 **bukan** karena threshold atau training. Struktur data membuat pasangan "close call" tidak ada — semua kandidat terpisah sempurna.

In [5]:
same_mask = pred_base["is_same_customer"] == 1
w_pos = pred_base.loc[same_mask, "match_weight"]
w_neg = pred_base.loc[~same_mask, "match_weight"]
gap = w_pos.min() - w_neg.max()

print(f"Pos weight min  : {w_pos.min():>8.2f}")
print(f"Pos weight max  : {w_pos.max():>8.2f}")
print(f"Neg weight max  : {w_neg.max():>8.2f}")
print(f"Neg weight min  : {w_neg.min():>8.2f}")
print(f"GAP             : {gap:>8.2f}  (band REVIEW hanya 4 poin)")

Pos weight min  :    39.58
Pos weight max  :    70.54
Neg weight max  :  -196.84
Neg weight min  :  -196.84
GAP             :   236.42  (band REVIEW hanya 4 poin)


## 5. Problem: Single Negative di Candidate Set

Baseline hanya menghasilkan 1 negative candidate. Dengan 1 negative, evaluasi precision/recall tidak bermakna (Precision 1.0 bisa hanya kebetulan).

**Dampak:**
- Threshold 0.20/0.85 redundan — tidak ada kandidat di zona transisi
- Evaluasi hanya mengukur bahwa duplikat terdeteksi, bukan akurasi di dunia riil
- Tidak ada kandidat yang perlu direview oleh manusia

In [6]:
con = duckdb.connect()
con.register("df", df)
rules_sql = {
    "phone+dob": "a.phone_main_std = b.phone_main_std AND a.dob_std = b.dob_std",
    "email+dob": "a.email_std = b.email_std AND a.dob_std = b.dob_std",
    "name+dob":  "a.first_name_std = b.first_name_std AND a.last_name_std = b.last_name_std AND a.dob_std = b.dob_std",
    "phone alone": "a.phone_main_std = b.phone_main_std",
}
q_tpl = "SELECT COUNT(*), SUM(CASE WHEN a.customer_id<>b.customer_id THEN 1 ELSE 0 END) FROM df a JOIN df b ON {c} AND a.unique_id < b.unique_id"

print("Blocking candidate counts (DuckDB SQL):")
for name, cond in rules_sql.items():
    t, d = con.execute(q_tpl.format(c=cond)).fetchone()
    print(f"  {name:14s}: {t:>10,} pairs | same-cid {t - (d or 0):>7,} | diff-cid {d or 0:>8,}")

Blocking candidate counts (DuckDB SQL):
  phone+dob     :      1,867 pairs | same-cid   1,867 | diff-cid        0
  email+dob     :      1,826 pairs | same-cid   1,826 | diff-cid        0
  name+dob      :      1,358 pairs | same-cid   1,357 | diff-cid        1
  phone alone   :      5,740 pairs | same-cid   1,867 | diff-cid    3,873


## 6. Improvement: Lebarkan Blocking (1 Komponen)

Perubahan **satu komponen** saja: tambah `block_on("phone_main_std")` ke blocking rules untuk prediksi.

**Alasan:**
1. Phone standardisasi kuat tapi tidak sempurna — banyak pasangan beda-customer share phone (bekas, nomor keluarga)
2. Menghasilkan 5.741 kandidat (vs 1.868 baseline) — cukup besar untuk evaluasi bermakna
3. 3.874 negative candidates memberikan evaluasi precision yang lebih realistis
4. Semua 1.867 reference positive masih ter-cover (recall blocking 100%)

**Apa yang TIDAK diubah:**
- Settings (comparisons, additional_columns_to_retain)
- Training steps (prior → u → m → EM)
- EM blocking rule (email_std)
- Decision thresholds (0.85 / 0.20)

## 7. Rebuild Model + Predict

In [7]:
RULES_RELAXED = [
    br.block_on("phone_main_std", "dob_std"),
    br.block_on("email_std", "dob_std"),
    br.block_on("first_name_std", "last_name_std", "dob_std"),
    br.block_on("device_id_std"),
    br.block_on("phone_main_std"),   # <-- ONLY change vs Nb04
]

settings = SettingsCreator(
    link_type="dedupe_only",
    blocking_rules_to_generate_predictions=RULES_RELAXED,
    comparisons=[
        cl.NameComparison("first_name_std"),
        cl.NameComparison("last_name_std"),
        cl.EmailComparison("email_std"),
        cl.LevenshteinAtThresholds("phone_main_std", [1, 2]),
        cl.DateOfBirthComparison("dob_std", input_is_string=True),
        cl.LevenshteinAtThresholds("address_std", 2),
        cl.LevenshteinAtThresholds("city_std", 1),
    ],
    retain_intermediate_calculation_columns=True,
    additional_columns_to_retain=["customer_id"],
)
linker = Linker(df, settings, db_api=DuckDBAPI(), set_up_basic_logging=False)
print("Linker (relaxed) created")

Linker (relaxed) created


In [8]:
linker.training.estimate_probability_two_random_records_match(
    ["l.device_id_std = r.device_id_std",
     "l.phone_main_std = r.phone_main_std and l.dob_std = r.dob_std"],
    recall=0.95,
)
print("Prior estimated")

Prior estimated


In [9]:
linker.training.estimate_u_using_random_sampling(max_pairs=1e7)
print("u parameters estimated")

u parameters estimated


In [10]:
labels_sdf = linker.table_management.register_labels_table(
    pd.DataFrame(
        [(i, j, "dedupe", "dedupe") for (i, j) in ref_pos],
        columns=["unique_id_l", "unique_id_r", "source_dataset_l", "source_dataset_r"],
    ),
    overwrite=True,
)
linker.training.estimate_m_from_pairwise_labels(labels_sdf)
print(f"m parameters estimated from {len(ref_pos):,} positive labels")

m parameters estimated from 1,867 positive labels


In [11]:
linker.training.estimate_parameters_using_expectation_maximisation(br.block_on("email_std"))
print("EM completed")

Level Levenshtein distance of phone_main_std <= 1 on comparison phone_main_std not observed in dataset, unable to train m value

Level Levenshtein distance of phone_main_std <= 2 on comparison phone_main_std not observed in dataset, unable to train m value

Level Exact match on date of birth on comparison dob_std not observed in dataset, unable to train m value

Level DamerauLevenshtein distance <= 1 on comparison dob_std not observed in dataset, unable to train m value

Level Abs date difference <= 1 month on comparison dob_std not observed in dataset, unable to train m value

Level Abs date difference <= 1 year on comparison dob_std not observed in dataset, unable to train m value

Level Abs date difference <= 10 year on comparison dob_std not observed in dataset, unable to train m value

Level All other comparisons on comparison dob_std not observed in dataset, unable to train m value

Level Levenshtein distance of address_std <= 2 on comparison address_std not observed in dataset, 

EM completed


In [12]:
results_relaxed = linker.inference.predict(threshold_match_probability=0.0)
pred = results_relaxed.as_pandas_dataframe()
pred["unique_id_l"] = pred["unique_id_l"].astype(int)
pred["unique_id_r"] = pred["unique_id_r"].astype(int)
pred["customer_id_l"] = df.set_index("unique_id")["customer_id"].loc[pred["unique_id_l"].values].values
pred["customer_id_r"] = df.set_index("unique_id")["customer_id"].loc[pred["unique_id_r"].values].values
pred["is_same_customer"] = (pred["customer_id_l"] == pred["customer_id_r"]).astype(int)
print(f"Predictions: {len(pred):,} candidate pairs")


 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'email_std':
    m values not fully trained
Comparison: 'email_std':
    u values not fully trained
Comparison: 'phone_main_std':
    m values not fully trained
Comparison: 'dob_std':
    m values not fully trained
Comparison: 'dob_std':
    u values not fully trained
Comparison: 'address_std':
    m values not fully trained
Comparison: 'address_std':
    u values not fully trained
Comparison: 'city_std':
    m values not fully trained


Predictions: 5,741 candidate pairs


## 8. Decision & Evaluasi

In [16]:
def decide(prob):
    if prob >= 0.85:
        return "MATCH"
    elif prob >= 0.20:
        return "REVIEW"
    else:
        return "NON-MATCH"

pred["decision"] = pred["match_probability"].apply(decide)
print(pred["decision"].value_counts().to_string())

decision
NON-MATCH    3874
MATCH        1867


In [17]:
tp = int(((pred["is_same_customer"] == 1) & (pred["decision"] == "MATCH")).sum())
fn = int(((pred["is_same_customer"] == 1) & (pred["decision"] != "MATCH")).sum())
tn = int(((pred["is_same_customer"] == 0) & (pred["decision"] == "NON-MATCH")).sum())
fp = int(((pred["is_same_customer"] == 0) & (pred["decision"] == "MATCH")).sum())
prec = tp / (tp + fp) if (tp + fp) else 0
rec  = tp / (tp + fn) if (tp + fn) else 0
f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0

print(f"TP {tp:,} | TN {tn:,} | FP {fp:,} | FN {fn:,}")
print(f"Precision {prec:.4f}  Recall {rec:.4f}  F1 {f1:.4f}")

TP 1,867 | TN 3,874 | FP 0 | FN 0
Precision 1.0000  Recall 1.0000  F1 1.0000


In [18]:
pd.crosstab(pred["is_same_customer"], pred["decision"], margins=True)

decision,MATCH,NON-MATCH,All
is_same_customer,,,
0,0,3874,3874
1,1867,0,1867
All,1867,3874,5741


In [19]:
pred[["match_weight", "match_probability", "decision"]].describe()

,match_weight,match_probability
count,5741.000000,5.741000e+03
mean,-85.048580,3.252047e-01
std,101.808613,4.684921e-01
min,-202.510178,1.092351e-61
25%,-156.323973,8.745710e-48
50%,-156.323973,8.745710e-48
75%,59.739242,1.000000e+00
max,70.319274,1.000000e+00


### Error Analysis

- **FP (0):** Tidak ada negative customer yang diprediksi sebagai MATCH. Semua hard negative (same phone, beda customer) berhasil dipisahkan oleh model.
- **FN (0):** Tidak ada positive customer yang terlewat. Semua duplikat asli terdeteksi.
- **REVIEW (0):** Masih kosong karena gap weight sangat lebar. Lihat diskusi di bawah.

## 9. Comparasi Baseline vs Relaxed

In [20]:
w = pred["match_weight"]
same_mask_r = pred["is_same_customer"] == 1
diff_mask_r = pred["is_same_customer"] == 0
gap_r = w[same_mask_r].min() - w[diff_mask_r].max()

baseline_stats = {
    "candidates":   len(pred_base),
    "negatives":    int((pred_base["is_same_customer"] == 0).sum()),
    "TP": 1867, "FP": 0, "TN": 1, "FN": 0,
    "Precision": 1.0, "Recall": 1.0, "F1": 1.0,
    "pos_min_w":  w_pos.min(),
    "neg_max_w":  w_neg.max(),
    "GAP": w_pos.min() - w_neg.max(),
}
relaxed_stats = {
    "candidates":   len(pred),
    "negatives":    int(diff_mask_r.sum()),
    "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    "Precision": round(prec, 4), "Recall": round(rec, 4), "F1": round(f1, 4),
    "pos_min_w":  w[same_mask_r].min(),
    "neg_max_w":  w[diff_mask_r].max(),
    "GAP": gap_r,
}

comparison = pd.DataFrame({
    "Metric": ["Candidate pairs", "Negative candidates", "TP", "FP", "TN", "FN",
               "Precision", "Recall", "F1", "Positive min weight", "Negative max weight", "GAP"],
    "Baseline (strict)": [baseline_stats["candidates"], baseline_stats["negatives"],
                          baseline_stats["TP"], baseline_stats["FP"], baseline_stats["TN"], baseline_stats["FN"],
                          baseline_stats["Precision"], baseline_stats["Recall"], baseline_stats["F1"],
                          baseline_stats["pos_min_w"], baseline_stats["neg_max_w"], baseline_stats["GAP"]],
    "Relaxed (+phone)": [relaxed_stats["candidates"], relaxed_stats["negatives"],
                         relaxed_stats["TP"], relaxed_stats["FP"], relaxed_stats["TN"], relaxed_stats["FN"],
                         relaxed_stats["Precision"], relaxed_stats["Recall"], relaxed_stats["F1"],
                         relaxed_stats["pos_min_w"], relaxed_stats["neg_max_w"], relaxed_stats["GAP"]],
})
comparison.set_index("Metric")

,Baseline (strict),Relaxed (+phone)
Metric,,
Candidate pairs,1868.000000,5741.000000
Negative candidates,1.000000,3874.000000
TP,1867.000000,1867.000000
FP,0.000000,0.000000
TN,1.000000,3874.000000
FN,0.000000,0.000000
Precision,1.000000,1.000000
Recall,1.000000,1.000000
F1,1.000000,1.000000


### Diskusi Perbandingan

**Kesimpulan kritis:**
1. Lebarkan blocking menghasilkan evaluasi yang **lebih realistis** (3.874 negatif diuji vs 1 sebelumnya)
2. Model tetap sempurna (FP=0, FN=0) karena phone std sangat diskriminatif
3. REVIEW tetap 0: semua kandidat relaksasi juga terpisah ekstrem (weight -127 s/d -260 untuk negatif)
4. **Gap weight** berkurang dari baseline — mengindikasikan tanda polarisasi lebih baik, tapi masih belum cukup untuk REVIEW
5. **Tindakan berikutnya:** perlu kandidat yang benar-benar ambigu — misalnya fuzzy blocking (partial match) untuk menciptakan "close calls" (lihat Nb06)

## 10. Entity Mapping

In [21]:
clusters_relaxed = linker.clustering.cluster_pairwise_predictions_at_threshold(
    results_relaxed, threshold_match_probability=0.80
)
clusters_df_r = clusters_relaxed.as_pandas_dataframe()
clusters_df_r.to_csv(ENTITY_RELAX, index=False)

print(f"Clusters (relaxed): {clusters_df_r['cluster_id'].nunique():,}")
print(f"Rows              : {len(clusters_df_r):,}")

Clusters (relaxed): 48,200
Rows              : 50,000


## 11. Visualisasi

In [22]:
linker.visualisations.match_weights_chart()

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\altair\vegalite\v6\api.py:4138: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [ ]:
records = results_relaxed.as_record_dict(limit=1)
linker.visualisations.waterfall_chart(records)

## 12. Summary & Rekomendasi

### Temuan Nb05

**Struktural (bukan bug):**
- REVIEW=0 disebabkan gap weight sangat lebar. Tidak ada kandidat "close call".
- Baseline (strict blocking) hanya menghasilkan 1 negative candidate → evaluasi tidak bermakna.
- Percobaan lebarkan blocking (phone_main_std) menghasilkan 3.874 negatif → model tetap FP=0 FN=0.

**Efektivitas perbaikan:**
- FP=0, FN=0 di relaxed → model berfungsi dengan sangat baik untuk data ini.
- REVIEW tetap kosong karena model terlalu memisahkan dengan tepat — ini kekuatan, bukan kelemahan.
- Evaluasi improved: precision/recall kini dihitung terhadap 3.874 negatif (bukan 1).

**Tantangan (perlu Nb06):**
1. **Gap weight terlalu besar** untuk data dengan kandidat eksklusif. Untuk menciptakan "human review zone" perlu kandidat yang benar-benar ambigu — misalnya kandidat dengan overlap parsial pada field fuzzy (name levenshtein tinggi, email mirip tapi beda suffix).
2. **Training m** belum mengamati level fuzzy (phone 1-edit, dob beda 1 tahun, dll) → semua berat di extreme.
3. **Threshold 0.20/0.85 redundan** — tidak perlu diperbaiki sampai ada kandidat yang benar-benar ambigu.

**Rekomendasi Nb06:**
- Bangun kandidat ambigu (fuzzy blocking atau dua field overlap parsial)
- Lihat apakah m terlatih lebih moderat pada kandidat beragam → gap weight menyusut
- Evaluasi threshold pada kandidat ambigu

---

*Nb05 selesai. Lanjut ke Nb06 untuk kandidat ambigu & threshold calibration.*